# Generate Supplementary Tables

This notebook programmatically generates the supplementary tables for the paper.

In [11]:
import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.stats as stats
import scikit_posthocs as sp
from sklearn.decomposition import PCA

In [12]:
# --- CONFIGURATION ---
# Load analysis configuration
with open("../configs/analysis.yaml", "r") as f:
    analysis_config = yaml.safe_load(f)

# Define constants from the config
MIN_NULL_WINDOWS = 15
Z_OUTLIER_THRESH = 20.0 
PROJECT_ROOT = Path("../")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACTS_DIR / "supplementary_tables"

# Create the output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

# Load the best HMM model
with open(ARTIFACTS_DIR / "chmm_8_full_dataset" / "best_model.json", "r") as f:
    hmm_model = json.load(f)

# --- HELPER FUNCTIONS ---

def get_regime_key(pT, pC, class_names):
    """Get the regime class name from therapist and client probabilities."""
    pT_idx = np.argmax(pT)
    pC_idx = np.argmax(pC)
    return f"T{class_names[pT_idx]}-C{class_names[pC_idx]}"

def weighted_mean(df, values, weights):
    """Compute the weighted mean and return a named Series."""
    weighted_sum = (df[values].T * df[weights]).T.sum()
    total_weight = df[weights].sum()
    return pd.Series(weighted_sum / total_weight, index=values)

def run_kruskal_dunn(data, metric, group_col="regime_class"):
    """Run Kruskal-Wallis and Dunn's post-hoc test."""
    # Kruskal-Wallis test
    groups = [group[metric].values for name, group in data.groupby(group_col)]
    h_stat, p_val = stats.kruskal(*groups)
    
    # Dunn's post-hoc test
    dunn_results = sp.posthoc_dunn(data, val_col=metric, group_col=group_col, p_adjust='bonferroni')
    
    return h_stat, p_val, dunn_results

## Table S1: HMM Regime Properties

This table shows the key properties of each of the 8 latent states (regimes) from the chosen cross-validated HMM. This includes the self-transition probability, the expected dwell time (in seconds), and the exit entropy, which measures the predictability of the next state.

In [13]:
# Extract HMM parameters
pT = np.array(hmm_model['pT'])
pC = np.array(hmm_model['pC'])
A = np.array(hmm_model['A'])
class_names = ["neg", "neu", "pos"] # This is based on the original analysis notebooks
n_states = A.shape[0]

# Create regime class names and a canonical ordering
regime_classes = [get_regime_key(pT[i, :], pC[i, :], class_names) for i in range(n_states)]
regime_order = sorted(regime_classes, key=lambda x: (x.split('-')[0], x.split('-')[1]))
regime_map = {name: i for i, name in enumerate(regime_classes)}
ordered_indices = [regime_map[name] for name in regime_order]

# Calculate HMM properties
s1_data = []
for i, state_idx in enumerate(ordered_indices):
    # Self-transition probability
    self_transition_prob = A[state_idx, state_idx]
    
    # Dwell time (in seconds, assuming 250ms per window)
    dwell_time = 1 / (1 - self_transition_prob) * 0.25
    
    # Exit entropy
    exit_probs = A[state_idx, np.arange(n_states) != state_idx]
    exit_probs /= exit_probs.sum() # Normalize to create a distribution
    exit_entropy = stats.entropy(exit_probs, base=2)
    
    s1_data.append({
        "Regime": regime_order[i],
        "Self-Transition Probability": self_transition_prob,
        "Dwell Time (s)": dwell_time,
        "Exit Entropy (bits)": exit_entropy
    })

s1_df = pd.DataFrame(s1_data)
s1_df.to_csv(OUTPUT_DIR / "S1_hmm_regime_properties.csv", index=False, float_format="%.3f")

print("Table S1: HMM Regime Properties")
display(s1_df)

Table S1: HMM Regime Properties


,Regime,Self-Transition Probability,Dwell Time (s),Exit Entropy (bits)
0,Tneg-Cneg,0.769984,1.086881,2.027454
1,Tneg-Cneu,0.745947,0.984048,1.993945
2,Tneu-Cneg,0.793453,1.210380,1.908484
3,Tneu-Cneu,0.884219,2.159252,2.179221
4,Tneu-Cneu,0.884219,2.159252,2.179221
5,Tpos-Cneg,0.699040,0.830676,1.924386
6,Tpos-Cneu,0.170798,0.301495,0.680229
7,Tpos-Cneu,0.170798,0.301495,0.680229


## Table S2: catRQA Regime Comparison

This table presents the results of statistical tests comparing three key catRQA metrics (Recurrence Rate, Determinism, and Laminarity) across the 8 HMM regimes. We use a Kruskal-Wallis H-test as an omnibus test for any difference across regimes. If significant, we follow up with Dunn's post-hoc tests (with Bonferroni correction) to identify which specific pairs of regimes differ. All metrics are Z-scored against a null distribution of 100 time-shuffled surrogates.

In [14]:
# Load catRQA episode results
catrqa_episode_df = pd.read_csv(ARTIFACTS_DIR / "joint_affect_catRQA" / "catrqa_episode_results.csv")

# Map regime state index to class name
catrqa_episode_df["regime_class"] = catrqa_episode_df["regime"].map(dict(enumerate(regime_classes)))

# Aggregate to dyad-regime level, weighted by duration
dyad_regime_catrqa = catrqa_episode_df.groupby(["dyad_id", "regime_class"]).apply(
    weighted_mean, values=["RR_Z_shuffle", "DET_Z_shuffle", "LAM_Z_shuffle"], weights="duration"
).reset_index()

# Filter out extreme outliers
dyad_regime_catrqa = dyad_regime_catrqa[
    (dyad_regime_catrqa["RR_Z_shuffle"].abs() < Z_OUTLIER_THRESH) &
    (dyad_regime_catrqa["DET_Z_shuffle"].abs() < Z_OUTLIER_THRESH) &
    (dyad_regime_catrqa["LAM_Z_shuffle"].abs() < Z_OUTLIER_THRESH)
]

# Run tests for each metric
s2_results = []
catrqa_metrics = ["RR_Z_shuffle", "DET_Z_shuffle", "LAM_Z_shuffle"]
metric_names = ["Recurrence Rate (Z)", "Determinism (Z)", "Laminarity (Z)"]

for metric, name in zip(catrqa_metrics, metric_names):
    h_stat, p_val, dunn_results = run_kruskal_dunn(dyad_regime_catrqa, metric)

    s2_results.append({
        "Metric": name,
        "Kruskal-Wallis H": h_stat,
        "Kruskal-Wallis p-value": p_val,
        "Dunn Results": dunn_results
    })

# Save and display results
s2_summary_list = []
for i, res in enumerate(s2_results):
    print(f"--- {res['Metric']} ---")
    print(f"Kruskal-Wallis H-statistic: {res['Kruskal-Wallis H']:.3f}, p-value: {res['Kruskal-Wallis p-value']:.3e}")
    print("Dunn's Post-Hoc (Bonferroni corrected p-values):")

    dunn_display = res["Dunn Results"].copy()
    dunn_display.index.name = "Regime 1"
    dunn_display.columns.name = "Regime 2"
    display(dunn_display.style.format("{:.3f}"))

    # For saving, stack the Dunn results into a tidy format
    dunn_tidy = (
        res["Dunn Results"]
        .rename_axis(index="Regime 1", columns="Regime 2")
        .stack()
        .reset_index(name="p-value")
    )
    dunn_tidy["Metric"] = res["Metric"]
    dunn_tidy["Kruskal-Wallis H"] = res["Kruskal-Wallis H"]
    dunn_tidy["Kruskal-Wallis p-value"] = res["Kruskal-Wallis p-value"]
    s2_summary_list.append(dunn_tidy)

s2_df = pd.concat(s2_summary_list, ignore_index=True)
s2_df = s2_df[["Metric", "Kruskal-Wallis H", "Kruskal-Wallis p-value", "Regime 1", "Regime 2", "p-value"]]
s2_df.to_csv(OUTPUT_DIR / "S2_catrqa_regime_comparison.csv", index=False, float_format="%.3f")

--- Recurrence Rate (Z) ---
Kruskal-Wallis H-statistic: 28.260, p-value: 1.105e-05
Dunn's Post-Hoc (Bonferroni corrected p-values):


/var/folders/fg/dc68zxv12sz5v55n18sqsvmw0000gn/T/ipykernel_13733/432616604.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dyad_regime_catrqa = catrqa_episode_df.groupby(["dyad_id", "regime_class"]).apply(


Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,1.000,0.000,0.000,0.305
Tneg-Cneu,1.000,1.000,0.031,0.034,1.000
Tneu-Cneg,0.000,0.031,1.000,1.000,0.312
Tneu-Cneu,0.000,0.034,1.000,1.000,0.333
Tpos-Cneg,0.305,1.000,0.312,0.333,1.000


--- Determinism (Z) ---
Kruskal-Wallis H-statistic: 13.734, p-value: 8.196e-03
Dunn's Post-Hoc (Bonferroni corrected p-values):


Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,0.057,0.029,0.016,1.000
Tneg-Cneu,0.057,1.000,1.000,1.000,1.000
Tneu-Cneg,0.029,1.000,1.000,1.000,1.000
Tneu-Cneu,0.016,1.000,1.000,1.000,1.000
Tpos-Cneg,1.000,1.000,1.000,1.000,1.000


--- Laminarity (Z) ---
Kruskal-Wallis H-statistic: 22.640, p-value: 1.494e-04
Dunn's Post-Hoc (Bonferroni corrected p-values):


Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,0.258,0.006,0.000,1.000
Tneg-Cneu,0.258,1.000,1.000,0.568,1.000
Tneu-Cneg,0.006,1.000,1.000,1.000,0.168
Tneu-Cneu,0.000,0.568,1.000,1.000,0.020
Tpos-Cneg,1.000,1.000,0.168,0.020,1.000


## Table S3: Grammar/Motif Statistics Comparison

This table is analogous to the catRQA table, but for metrics derived from the symbolic grammar analysis (motifs). We compare metrics for motif complexity, diversity (entropy), and frequency across the 8 HMM regimes for motifs of length k=2 and k=3. Again, we use Kruskal-Wallis omnibus tests followed by Dunn's post-hoc tests where appropriate. All metrics are Z-scored against time-shuffled surrogates.

In [15]:
# Load grammar episode results
grammar_episode_df = pd.read_csv(ARTIFACTS_DIR / "joint_affect_grammar" / "grammar_episode_results.csv")

# Filter data
grammar_episode_df = grammar_episode_df[
    grammar_episode_df["k"].isin([2, 3]) &
    (grammar_episode_df["n_windows"] >= MIN_NULL_WINDOWS)
].copy()

# Map regime state index to class name
grammar_episode_df["regime_class"] = grammar_episode_df["regime"].map(dict(enumerate(regime_classes)))

# Aggregate to dyad-regime level using available shuffle-z columns
shuffle_z_cols = [c for c in grammar_episode_df.columns if c.endswith("_z_shuffle")]
dyad_regime_grammar = grammar_episode_df.groupby(["dyad_id", "regime_class", "k"]).apply(
    weighted_mean, values=shuffle_z_cols, weights="duration"
).reset_index()

# Run tests for each metric and k
s3_results = []
grammar_metrics = [
    "effective_motif_count_z_shuffle",
    "motif_entropy_z_shuffle",
    "motif_transition_diversity_z_shuffle",
]
metric_names = [
    "Effective Motif Count (Z)",
    "Motif Entropy (Z)",
    "Motif Transition Diversity (Z)",
]

for k_val in [2, 3]:
    for metric, name in zip(grammar_metrics, metric_names):
        data_subset = dyad_regime_grammar[dyad_regime_grammar["k"] == k_val]
        h_stat, p_val, dunn_results = run_kruskal_dunn(data_subset, metric)

        s3_results.append({
            "k": k_val,
            "Metric": name,
            "Kruskal-Wallis H": h_stat,
            "Kruskal-Wallis p-value": p_val,
            "Dunn Results": dunn_results
        })

# Save and display results
s3_summary_list = []
for res in s3_results:
    print(f"--- k={res['k']}, {res['Metric']} ---")
    print(f"Kruskal-Wallis H-statistic: {res['Kruskal-Wallis H']:.3f}, p-value: {res['Kruskal-Wallis p-value']:.3e}")
    print("Dunn's Post-Hoc (Bonferroni corrected p-values):")

    dunn_display = res["Dunn Results"].copy()
    dunn_display.index.name = "Regime 1"
    dunn_display.columns.name = "Regime 2"
    display(dunn_display.style.format("{:.3f}"))

    # For saving, stack the Dunn results into a tidy format
    dunn_tidy = (
        res["Dunn Results"]
        .rename_axis(index="Regime 1", columns="Regime 2")
        .stack()
        .reset_index(name="p-value")
    )
    dunn_tidy["k"] = res["k"]
    dunn_tidy["Metric"] = res["Metric"]
    dunn_tidy["Kruskal-Wallis H"] = res["Kruskal-Wallis H"]
    dunn_tidy["Kruskal-Wallis p-value"] = res["Kruskal-Wallis p-value"]
    s3_summary_list.append(dunn_tidy)

s3_df = pd.concat(s3_summary_list, ignore_index=True)
s3_df = s3_df[["k", "Metric", "Kruskal-Wallis H", "Kruskal-Wallis p-value", "Regime 1", "Regime 2", "p-value"]]
s3_df.to_csv(OUTPUT_DIR / "S3_grammar_regime_comparison.csv", index=False, float_format="%.3f")

--- k=2, Effective Motif Count (Z) ---
Kruskal-Wallis H-statistic: 10.218, p-value: 3.691e-02
Dunn's Post-Hoc (Bonferroni corrected p-values):


/var/folders/fg/dc68zxv12sz5v55n18sqsvmw0000gn/T/ipykernel_13733/1722558098.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dyad_regime_grammar = grammar_episode_df.groupby(["dyad_id", "regime_class", "k"]).apply(


Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,0.327,1.000,1.000,1.000
Tneg-Cneu,0.327,1.000,0.037,1.000,1.000
Tneu-Cneg,1.000,0.037,1.000,0.960,0.889
Tneu-Cneu,1.000,1.000,0.960,1.000,1.000
Tpos-Cneg,1.000,1.000,0.889,1.000,1.000


--- k=2, Motif Entropy (Z) ---
Kruskal-Wallis H-statistic: 9.657, p-value: 4.662e-02
Dunn's Post-Hoc (Bonferroni corrected p-values):


Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,0.414,1.000,1.000,1.000
Tneg-Cneu,0.414,1.000,0.054,1.000,1.000
Tneu-Cneg,1.000,0.054,1.000,0.953,0.822
Tneu-Cneu,1.000,1.000,0.953,1.000,1.000
Tpos-Cneg,1.000,1.000,0.822,1.000,1.000


...

Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,1.000,0.605,1.000,1.000
Tneg-Cneu,1.000,1.000,0.001,0.399,1.000
Tneu-Cneg,0.605,0.001,1.000,0.276,0.026
Tneu-Cneu,1.000,0.399,0.276,1.000,1.000
Tpos-Cneg,1.000,1.000,0.026,1.000,1.000


--- k=3, Motif Entropy (Z) ---
Kruskal-Wallis H-statistic: 20.748, p-value: 3.553e-04
Dunn's Post-Hoc (Bonferroni corrected p-values):


Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,1.000,0.416,1.000,1.000
Tneg-Cneu,1.000,1.000,0.001,0.648,1.000
Tneu-Cneg,0.416,0.001,1.000,0.074,0.014
Tneu-Cneu,1.000,0.648,0.074,1.000,1.000
Tpos-Cneg,1.000,1.000,0.014,1.000,1.000


--- k=3, Motif Transition Diversity (Z) ---
Kruskal-Wallis H-statistic: 21.770, p-value: 2.227e-04
Dunn's Post-Hoc (Bonferroni corrected p-values):


Regime 2,Tneg-Cneg,Tneg-Cneu,Tneu-Cneg,Tneu-Cneu,Tpos-Cneg
Regime 1,,,,,
Tneg-Cneg,1.000,1.000,0.602,1.000,1.000
Tneg-Cneu,1.000,1.000,0.001,0.007,1.000
Tneu-Cneg,0.602,0.001,1.000,1.000,0.038
Tneu-Cneu,1.000,0.007,1.000,1.000,0.157
Tpos-Cneg,1.000,1.000,0.038,0.157,1.000
